# 🧪 Fine-Tuning Eficiente de um Modelo de Linguagem com LoRA (PEFT)

Este tutorial conduz você, passo a passo, pelo processo de **ajuste fino (fine-tuning)** de um modelo de linguagem causal (ex.: `distilgpt2`) utilizando **LoRA** (*Low-Rank Adaptation*), uma das técnicas de **PEFT** (*Parameter-Efficient Fine-Tuning*).  
Ao final, você compreenderá os fundamentos matemáticos por trás do método e saberá como adaptar um modelo grande com muito menos recursos computacionais.

**O que você vai aprender:**
- A diferença entre *full fine-tuning* e *PEFT*.
- A ideia central do LoRA (matrizes de baixo posto).
- Como preparar dados, configurar, treinar e testar um modelo com LoRA.
- Comparar a qualidade das respostas antes e depois do ajuste fino.

## 📚 1. Por que Fine-Tuning Eficiente?

Modelos de linguagem modernos possuem bilhões de parâmetros. Atualizar **todos** os pesos durante o treinamento (*full fine-tuning*) exige:
- GPUs com dezenas de GB de memória.
- Armazenamento de uma cópia completa do modelo para cada tarefa.

**PEFT (Parameter-Efficient Fine-Tuning)** resolve esse problema treinando apenas um pequeno conjunto de **novos parâmetros**, mantendo o modelo base congelado.  

### 🔹 LoRA (Low-Rank Adaptation)
A hipótese do LoRA é que as atualizações dos pesos durante o fine-tuning possuem uma **estrutura de baixo posto** (*low intrinsic rank*).  
Assim, em vez de aprender a matriz completa de atualização $\Delta W \in \mathbb{R}^{d \times k}$, aprendemos duas matrizes menores:

$$\Delta W = B \cdot A$$

onde:
- $B \in \mathbb{R}^{d \times r}$
- $A \in \mathbb{R}^{r \times k}$
- $r \ll \min(d, k)$ (o **rank** da adaptação)

O número de parâmetros treináveis cai de $d \times k$ para $r \times (d + k)$, uma redução drástica quando $r$ é pequeno.

### 🔹 Como isso é usado na prática?
Durante o treinamento, a saída de uma camada linear original $h = W x$ é modificada para:

$$h = W x + \Delta W x = W x + B A x$$

A matriz $A$ é inicializada com uma distribuição gaussiana e $B$ com zeros, de forma que no início $\Delta W = 0$.  
Um fator de escala $\alpha$ controla a intensidade da adaptação; frequentemente a atualização é escalada por $\frac{\alpha}{r}$:

$$h = W x + \frac{\alpha}{r} B A x$$

Após o treinamento, podemos **fundir** (*merge*) os pesos adaptados ao modelo original: $W_{\text{merged}} = W + \frac{\alpha}{r} BA$, eliminando qualquer custo extra na inferência.

Neste notebook, usaremos a biblioteca `peft` (Hugging Face) para aplicar LoRA ao `distilgpt2`.

## 📦 2. Requisitos

Execute o comando abaixo para instalar as dependências necessárias (descomente a linha caso ainda não estejam instaladas):

In [ ]:
!pip install transformers datasets peft accelerate torch

Importe os módulos que serão utilizados ao longo do processo:

In [ ]:
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)
import torch

## 🔢 3. Carregar e Preparar o Dataset

Utilizaremos um arquivo `dataset.jsonl` onde cada linha contém uma instrução (`instruction`) e a saída desejada (`output`).  
Vamos converter cada exemplo em uma única string no formato:
```
Instruction: <instrução>
Output: <saída>
```
e depois dividir o conjunto em treino (80%) e validação (20%).

In [ ]:
from datasets import load_dataset

# 1. Carrega o arquivo original (sem aplicar nenhuma função de texto único)
dataset_raw = load_dataset('json', data_files='dataset.jsonl')

# 2. Divide direto em treino e teste mantendo as colunas originais separadas
dataset = dataset_raw["train"].train_test_split(test_size=0.2, seed=42)
print("Dataset Seq2Seq pronto (colunas separadas):", dataset)

## 🤖 4. Carregar o Modelo Pré-Treinado e o Tokenizador

Vamos carregar o `distilgpt2` – uma versão menor e mais rápida do GPT-2, ideal para experimentação.  
Como o tokenizador original não define um `pad_token`, usaremos o `eos_token` no lugar.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

model_id = "facebook/bart-base"

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForSeq2SeqLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

## 🧪 5. Inferência ANTES do Fine-Tuning

Antes de qualquer treinamento, vamos ver como o modelo base responde a uma pergunta que está no nosso dataset.  
Isso servirá como **linha de base** para compararmos com o modelo ajustado.

In [ ]:
def generate_response_seq2seq(model, tokenizer, instruction, input_text=""):
    """Gera uma resposta direta para modelos Seq2Seq."""
    if input_text:
        prompt = f"Instruction: {instruction} Input: {input_text}"
    else:
        prompt = f"Instruction: {instruction}"
    
    # Enviando os tensores para a GPU
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=64, # Modelos seq2seq tendem a ser mais diretos
        do_sample=True,          
        temperature=0.7
    )
    
    # 🌟 MUDANÇA: O decoder já traz apenas a resposta, sem o prompt junto!
    resposta = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return resposta.strip()

# Teste idêntico para a tabela comparativa:
test_instruction = "O que é o sistema e-gestor e qual a sua utilidade para os gestores do SUS?"
test_input = "A disponibilização de sistema de credenciamento on-line via sistema e-gestor é uma medida de desburocratização para aprimorar e facilitar o acompanhamento das solicitações pelos gestores, conselhos de saúde, profissionais e usuários do SUS."

print("=== ANTES DO FINE-TUNING (SEQ2SEQ) ===")
print(f"Instrução: {test_instruction}")
print(f"Resposta base: {generate_response_seq2seq(model, tokenizer, test_instruction, test_input)}")

> **Observação:** O modelo base provavelmente gerará um texto genérico ou sem relação direta com a instrução, pois ainda não foi adaptado ao nosso domínio.

## ✂️ 6. Tokenização do Dataset

Transformamos os textos em sequências de tokens que o modelo pode processar.  
Usaremos `padding="max_length"` e `truncation=True` para garantir que todas as amostras tenham o mesmo comprimento (128 tokens).

In [ ]:
def tokenize_function_seq2seq(examples):
    # 1. Monta a entrada combinando a instrução e o contexto (Input)
    inputs = [
        f"Instruction: {inst}\nInput: {inp}" if inp else f"Instruction: {inst}"
        for inst, inp in zip(examples["instruction"], examples["input"])
    ]
    targets = examples["output"]
    
    # 2. Tokeniza a entrada (Encoder) e a resposta (Decoder)
    model_inputs = tokenizer(inputs, max_length=512, truncation=True)
    labels = tokenizer(text_target=targets, max_length=256, truncation=True)
    
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# 🌟 CORREÇÃO CRUCIAL: Pegamos os nomes das colunas de dentro da partição de treino
colunas_originais = dataset["train"].column_names

# Aplica a tokenização limpando os textos brutos antigos do 'train' e 'test'
tokenized_datasets = dataset.map(
    tokenize_function_seq2seq, 
    batched=True,
    remove_columns=colunas_originais
)

# Verifique se aparecem as duas partições prontas com as chaves numéricas
print("Dataset Seq2Seq Tokenizado:", tokenized_datasets)

## 🔧 7. Preparar o Modelo para LoRA

A função `prepare_model_for_kbit_training` ativa técnicas como *gradient checkpointing* e ajusta a arquitetura para treinamento eficiente.  
É essencial quando se utiliza quantização (QLoRA), mas também é recomendada mesmo sem quantização para melhor gerenciamento de memória.

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

# 1. Configuração do LoRA para Modelos Seq2Seq (Encoder-Decoder)
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],                 # Módulos-alvo dinâmicos de acordo com a arquitetura
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM          # 🌟 Crucial: muda para o paradigma Seq2Seq
)

# 2. Prepara e envelopa o modelo com os adaptadores LoRA
model = get_peft_model(model, lora_config)

# Mostra a drástica redução de parâmetros treináveis
model.print_trainable_parameters()

## 🧱 9. Data Collator para Modelagem Causal

O `DataCollatorForLanguageModeling` prepara os lotes para o treinamento de linguagem causal (sem *masked language modeling*).  
Ele automaticamente desloca os rótulos para que a tarefa seja prever o próximo token.

In [ ]:
from transformers import DataCollatorForSeq2Seq

# Gerencia o padding dinâmico da pergunta e da resposta separadamente
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

## ⚙️ 10. Argumentos de Treinamento

Definimos os hiperparâmetros do treinamento.  
Como nosso dataset é pequeno, usaremos 100 épocas e uma taxa de aprendizado relativamente alta (`1e-3`).  
O `eval_steps` controla a frequência da avaliação no conjunto de validação.

In [ ]:
training_args = TrainingArguments(
    output_dir="./resultados_bart",   # Altere para o nome do modelo (ex: ./resultados_t5)
    eval_strategy="epoch",
    learning_rate=3e-4,                  # Modelos Seq2Seq toleram uma taxa ligeiramente maior
    per_device_train_batch_size=4,       # Podemos usar 4 porque o modelo é super leve
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=2,       # Acumula gradientes para simular o mesmo lote real de 8 (4x2)
    num_train_epochs=5,                  # 5 épocas ajudam modelos menores a convergirem melhor
    weight_decay=0.01,
    fp16=True,                           # OBRIGATÓRIO: Mantém o treino extremamente veloz
    logging_steps=5,                     # Registra o Loss de forma granular para o gráfico
    save_strategy="epoch",               # Salva o progresso por época
    report_to="none",
)

## 🏋️ 11. Inicializar o Trainer

O `Trainer` do Hugging Face orquestra todo o ciclo de treinamento, avaliação e salvamento.

In [ ]:
# === SEÇÃO 10 ===
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],  # 🌟 Apenas o treino Seq2Seq
    eval_dataset=tokenized_datasets["test"],    # 🌟 Apenas o teste Seq2Seq
    data_collator=data_collator,
)

## 🚀 12. Treinar o Modelo

Iniciamos o treinamento. Acompanhe a perda (*loss*) nos logs – ela deve diminuir ao longo das épocas.

In [ ]:
trainer.train()

## 💾 13. Salvar o Modelo Ajustado e o Tokenizador

Ao final do treinamento, salvamos os pesos LoRA (apenas os adaptadores) e o tokenizador.

In [ ]:
# Define o caminho exclusivo para o BART
caminho_salvamento = "./modelo_final_bart"

print(f"Salvando adaptadores LoRA em: {caminho_salvamento}...")
trainer.save_model(caminho_salvamento)

print("Salvando configurações do tokenizador...")
tokenizer.save_pretrained(caminho_salvamento)

print("✅ BART salvo com sucesso e pronto para a API!")

## 💻 14. Inferência APÓS o Fine-Tuning

Agora carregamos o modelo ajustado e comparamos sua resposta com a versão base, usando **exatamente a mesma instrução**.

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

# 🌟 Correção: Carrega usando a classe correta para modelos Seq2Seq
finetuned_model = AutoModelForSeq2SeqLM.from_pretrained("./modelo_final_bart")
finetuned_tokenizer = AutoTokenizer.from_pretrained("./modelo_final_bart")

if finetuned_tokenizer.pad_token is None:
    finetuned_tokenizer.pad_token = finetuned_tokenizer.eos_token

print("=== DEPOIS DO FINE-TUNING (Bart) ===")
print(f"Instrução: {test_instruction}")
print(f"Resposta ajustada: {generate_response_seq2seq(finetuned_model, finetuned_tokenizer, test_instruction)}")

## 📊 15. Comparação e Conclusão

- **Antes do fine-tuning:** o modelo base não conhecia nosso domínio; sua resposta era genérica ou incoerente.
- **Depois do fine-tuning:** com apenas uma fração dos parâmetros treinados (via LoRA), o modelo aprendeu a estrutura desejada e gera respostas alinhadas com os exemplos fornecidos.

Esse é o poder do **PEFT**: adaptar grandes modelos de forma rápida, barata e com resultados surpreendentes.

### 📌 Resumo dos conceitos-chave

| Conceito | Descrição |
|----------|-----------|
| **Full fine-tuning** | Atualiza todos os pesos do modelo. |
| **PEFT** | Atualiza apenas um pequeno número de parâmetros novos. |
| **LoRA** | Decompõe a atualização $\Delta W$ em $B A$, com $r \ll \min(d,k)$. |
| **r** | Posto da decomposição – controla a capacidade da adaptação. |
| **$\alpha$** | Fator de escala que ajusta a intensidade da adaptação. |
| **Target modules** | Camadas onde os adaptadores LoRA são inseridos. |

Agora você pode experimentar com outros valores de `r`, `lora_alpha`, ou até mesmo outros modelos!